In [2]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import numpy as np
from gensim.models import Word2Vec
import fasttext.util
import fasttext
import fasttext.util
import gzip
import os
import pandas as pd
import anthropic
from tqdm import tqdm
from typing import List, Dict, Any
import json
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
from datetime import datetime
import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # FutureWarning 제거

In [28]:
df = pd.read_excel('../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../data/info.csv')
api_key = api.loc[0][1]

In [29]:
import os
import json
import pandas as pd
import numpy as np
import logging
import re
from typing import List, Dict, Any
import fasttext
import anthropic
import nest_asyncio
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio

pd.options.mode.chained_assignment = None


In [5]:
null_status = {}
for idx,col in enumerate(df.columns):    
    null_status[col] = round(df[col].notna().sum() / len(df),2)

pd.DataFrame(null_status,index=[0]).T

,0
환자번호,1.00
날짜,1.00
CC,1.00
약,0.21
장치,0.40
습관,0.57
찜질,0.60
"마사지, 스트레칭",0.21
PI,0.62
CMO,0.91


In [6]:
import asyncio
import json
import re
import logging
import pandas as pd
import anthropic  # anthropic 패키지가 설치되어 있어야 합니다.

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# API 호출 및 모델 관련 설정
class Config:
    SEMAPHORE_LIMIT = 3  # 동시 요청 수 감소
    MODEL_NAME = "claude-3-opus-20240229"
    MAX_TOKENS = 1000    # 토큰 수 제한
    TEMPERATURE = 0
    TOP_P = 1
    RATE_LIMIT_DELAY = 15  # 요청 간 지연 시간(초)
    MAX_RETRIES = 5

class MedicalCategoryRecommender:
    def __init__(self, api_key: str, df: pd.DataFrame):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.semaphore = asyncio.Semaphore(Config.SEMAPHORE_LIMIT)
        self.df = df

    async def make_api_call(self, prompt: str) -> dict:
        async with self.semaphore:
            response = await asyncio.to_thread(
                self.client.messages.create,
                model=Config.MODEL_NAME,
                max_tokens=Config.MAX_TOKENS,
                temperature=Config.TEMPERATURE,
                top_p=Config.TOP_P,
                messages=[{"role": "user", "content": prompt}]
            )
            content = response.content[0].text
            result = self._validate_and_parse_json(content)
            if not result:
                raise ValueError("Invalid JSON response")
            return result

    def _validate_and_parse_json(self, content: str) -> dict:
        """
        응답 텍스트에서 JSON 객체를 추출합니다.
        """
        try:
            # 우선 직접 파싱 시도
            json_obj = json.loads(content)
            return json_obj
        except json.JSONDecodeError:
            # JSON 파싱에 실패하면 정규표현식으로 추출 시도
            pattern = r'\{.*\}'
            match = re.search(pattern, content, re.DOTALL)
            if match:
                try:
                    json_obj = json.loads(match.group())
                    return json_obj
                except json.JSONDecodeError:
                    logger.error("정규표현식으로 추출한 JSON 파싱 실패")
                    return {}
            logger.error("JSON 파싱 실패. Content: " + content[:500])
            return {}

    def _random_sample(self, column_data: pd.Series, sample_size: int = 5000) -> list:
        """
        각 컬럼에서 Null이 아닌 데이터만 대상으로 랜덤하게 sample_size 개의 데이터를 추출합니다.
        """
        # Null이 아닌 데이터만 필터링
        valid_data = column_data.dropna()
        
        # 유효한 데이터가 없는 경우
        if len(valid_data) == 0:
            logger.warning(f"컬럼 '{column_data.name}'에 유효한 데이터가 없습니다.")
            return []
        
        # 유효한 데이터가 sample_size보다 적은 경우
        if len(valid_data) < sample_size:
            logger.info(f"컬럼 '{column_data.name}'의 유효한 데이터가 {len(valid_data)}개로, 요청된 sample_size({sample_size})보다 적습니다.")
            return valid_data.tolist()
        
        # 요청된 sample_size만큼 랜덤 샘플링
        return valid_data.sample(n=sample_size, random_state=42).tolist()

    def _generate_prompt_for_column(self, column_name: str, sample_data: list) -> str:
        """
        컬럼명과 해당 컬럼의 샘플 데이터를 기반으로 카테고리 추천 프롬프트를 생성합니다.
        """
        joined_data = "\n".join([str(item) for item in sample_data])
        prompt = f"""
            당신은 데이터 분석 전문가이자, 카테고리 분류 전문가입니다.
            아래는 '{column_name}' 컬럼의 샘플 데이터입니다:
            {joined_data}

            이 데이터를 바탕으로 '{column_name}' 컬럼을 효과적으로 분류할 수 있는 유의미한 카테고리(또는 범주)를 추천해 주세요.
            각 카테고리에 대해 간단한 설명도 함께 제공해 주시고, 반드시 아래 JSON 형식으로 응답해 주세요.

            응답 예시:
            {{
            "column": "{column_name}",
            "categories": [
                {{
                    "name": "예시 카테고리 1",
                    "description": "카테고리 1에 대한 설명"
                }},
                {{
                    "name": "예시 카테고리 2",
                    "description": "카테고리 2에 대한 설명"
                }}
                // 필요에 따라 더 추가
            ]
            }}
        """
        return prompt

    async def _get_recommendation(self, column: str, prompt: str):
        """
        특정 컬럼에 대해 API 호출하여 추천 결과를 받습니다.
        """
        try:
            result = await self.make_api_call(prompt)
            return (column, result)
        except Exception as e:
            logger.error(f"컬럼 {column} 처리 중 오류 발생: {e}")
            return (column, {})


    async def recommend_categories_for_all_columns(self) -> dict:
        from tqdm import tqdm  # 이렇게 수정
        """
        DataFrame의 모든 컬럼에 대해 추천 카테고리를 받아옵니다.
        """
        recommendations = {}
        tasks = []
        
        for column in tqdm(self.df.columns, desc="컬럼 처리 중"):  # tqdm.tqdm 대신 tqdm 사용
            # 샘플 데이터 추출
            sample_data = self._random_sample(self.df[column], sample_size=10)
            
            # 유효한 샘플이 없는 경우 스킵
            if not sample_data:
                logger.warning(f"컬럼 '{column}'에서 유효한 샘플을 추출할 수 없습니다.")
                continue
                
            try:
                prompt = self._generate_prompt_for_column(column, sample_data)
                tasks.append(self._get_recommendation(column, prompt))
                
            except Exception as e:
                logger.error(f"'{column}' 컬럼 처리 준비 중 오류 발생: {str(e)}")
                continue
        
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        for col, result in results:
            if isinstance(result, Exception):
                logger.error(f"'{col}' 컬럼 처리 중 오류 발생: {str(result)}")
                continue
            recommendations[col] = result
        
        return recommendations

# 예제 실행 코드
async def main():
    # 예시 DataFrame 생성: 실제 데이터로 대체하세요.
    columns = ['날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
               'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
               'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
               'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
               'Lateral excursion Protrusive excursion', 'End feel', '치료계획',
               'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견']
    
    # 각 컬럼에 대해 간단한 예시 데이터 생성
    # 실제 API 키를 입력하세요.
    recommender = MedicalCategoryRecommender(api_key, df)
    recommendations = await recommender.recommend_categories_for_all_columns()
    
    # 추천 결과 출력 (각 컬럼에 대해 추천받은 카테고리 JSON)
    print(json.dumps(recommendations, indent=2, ensure_ascii=False))
    
async def main():
    # ... 기존 코드 ...
    recommender = MedicalCategoryRecommender(api_key, df)
    recommendations = await recommender.recommend_categories_for_all_columns()
    
    # 결과를 데이터프레임으로 변환
    categories_data = []
    for column, result in recommendations.items():
        if 'categories' in result:
            for category in result['categories']:
                categories_data.append({
                    'column': column,
                    'category_name': category['name'],
                    'description': category['description']
                })
    
    # 데이터프레임 생성
    categories_df = pd.DataFrame(categories_data)
    
    # 결과 출력
    print("카테고리 추천 결과:")
    display(categories_df)
    
    # JSON 형식으로도 저장
    categories_df.to_json('category_recommendations.json', 
                         orient='records', 
                         force_ascii=False, 
                         indent=2)
    return categories_df

if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    categories_df = await main()


컬럼 처리 중: 100%|██████████| 32/32 [00:00<00:00, 204.38it/s]
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 429 Too Many Requests"
INFO:anthropic._base_client:Retrying request to /v1/messages in 0.434597 seconds
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 429 Too Many Requests"
INFO:anthropic._base_client:Retrying request to /v1/messages in 0.481935 seconds
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 429 Too Many Requests"
INFO:anthropic._base_client:Retrying request to /v1/messages in 0.410311 seconds
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 429 Too Many Requests"
INFO:anthropic._base_client:Retrying request to /v1/messages in 0.911555 seconds
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 429 Too Many Requests"
INFO:anthropic._base_client:Retrying request to /v1/messages in 0.784673 seconds
INFO:httpx:HTTP Request: POST ht

CancelledError: 

INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 429 Too Many Requests"
INFO:anthropic._base_client:Retrying request to /v1/messages in 0.914129 seconds
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 429 Too Many Requests"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


In [30]:
df = df.iloc[:,2:]

In [32]:
import asyncio
import json
import logging
import pandas as pd
import anthropic
from tenacity import retry, stop_after_attempt, wait_exponential
import time
import re
from tqdm.asyncio import tqdm_asyncio
from typing import List, Dict, Any

# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('medical_recommender.log')
    ]
)
logger = logging.getLogger(__name__)


class Config:
    SEMAPHORE_LIMIT: int = 3
    MODEL_NAME: str = "claude-3-opus-20240229"
    MAX_TOKENS: int = 1000
    TEMPERATURE: int = 0
    TOP_P: int = 1
    RATE_LIMIT_DELAY: int = 15
    MAX_RETRIES: int = 5
    SAMPLE_SIZE: int = 100


class MedicalCategoryRecommender:
    def __init__(self, api_key: str, df: pd.DataFrame):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.semaphore = asyncio.Semaphore(Config.SEMAPHORE_LIMIT)
        self.df = df
        self.last_request_time: float = 0

    def _get_valid_samples(self, column_data: pd.Series) -> List[Any]:
        """
        유효한(non-null, 공백이 아닌) 데이터만 추출하여 SAMPLE_SIZE 만큼 샘플링.
        """
        valid_data = column_data.dropna().replace('', pd.NA).dropna()
        if valid_data.empty:
            logger.warning(f"컬럼 '{column_data.name}'에 유효한 데이터가 없습니다.")
            return []
        if len(valid_data) <= Config.SAMPLE_SIZE:
            logger.info(f"컬럼 '{column_data.name}'의 전체 유효 데이터 {len(valid_data)}개 사용")
            return valid_data.tolist()
        return valid_data.sample(n=Config.SAMPLE_SIZE, random_state=42).tolist()

    @retry(
        stop=stop_after_attempt(Config.MAX_RETRIES),
        wait=wait_exponential(multiplier=1, min=4, max=10)
    )
    async def make_api_call(self, prompt: str) -> Dict[str, Any]:
        """
        API 호출 전 세마포어와 rate limit을 적용하여 클로드 API를 호출합니다.
        """
        async with self.semaphore:
            # Rate limiting: 마지막 호출 이후 Config.RATE_LIMIT_DELAY 초가 지나지 않았다면 대기
            current_time = time.time()
            elapsed = current_time - self.last_request_time
            if elapsed < Config.RATE_LIMIT_DELAY:
                wait_time = Config.RATE_LIMIT_DELAY - elapsed
                logger.debug(f"Rate limit 적용: {wait_time:.1f}초 대기")
                await asyncio.sleep(wait_time)

            try:
                response = await asyncio.to_thread(
                    self.client.messages.create,
                    model=Config.MODEL_NAME,
                    max_tokens=Config.MAX_TOKENS,
                    temperature=Config.TEMPERATURE,
                    top_p=Config.TOP_P,
                    system="JSON 형식으로만 응답하세요.",
                    messages=[{"role": "user", "content": prompt}]
                )
                self.last_request_time = time.time()
                content = response.content[0].text
                result = self._validate_and_parse_json(content)
                if not result:
                    raise ValueError("Invalid JSON response")
                return result

            except Exception as e:
                if "rate_limit_error" in str(e).lower():
                    logger.warning(f"Rate limit에 걸렸습니다. {Config.RATE_LIMIT_DELAY}초 후 재시도합니다.")
                    await asyncio.sleep(Config.RATE_LIMIT_DELAY)
                logger.exception("API 호출 중 오류 발생")
                raise

    def _validate_and_parse_json(self, content: str) -> Dict[str, Any]:
        """
        응답 내용을 직접 파싱하거나 정규식을 이용해 JSON 객체를 추출합니다.
        """
        logger.debug(f"응답 내용 파싱 시도 (일부): {content[:200]}...")
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            logger.warning("직접 JSON 파싱 실패. 정규식을 이용해 추출을 시도합니다.")
            try:
                # 단순한 JSON 객체 추출 패턴 (중첩 JSON 지원은 제한적)
                json_pattern = r'\{.*\}'
                match = re.search(json_pattern, content, re.DOTALL)
                if match:
                    json_str = match.group()
                    result = json.loads(json_str)
                    logger.info("정규식 추출로 JSON 파싱 성공")
                    return result
                else:
                    logger.error("유효한 JSON 객체를 찾지 못했습니다.")
                    return {"error": "유효한 JSON 없음", "content": content[:200]}
            except Exception as e:
                logger.exception("정규식으로 JSON 추출 실패")
                return {"error": str(e), "content": content[:200]}

    @retry(
        stop=stop_after_attempt(Config.MAX_RETRIES),
        wait=wait_exponential(multiplier=1, min=4, max=10)
    )
    async def _get_recommendation(self, column: str, prompt: str) -> (str, Dict[str, Any]):
        """
        특정 컬럼에 대해 API 호출하여 추천 결과를 반환합니다.
        """
        try:
            result = await self.make_api_call(prompt)
            return column, result
        except Exception as e:
            logger.exception(f"컬럼 '{column}' 처리 중 오류 발생")
            return column, {}

    async def recommend_categories_for_all_columns(self) -> Dict[str, Any]:
        """
        DataFrame의 각 컬럼에 대해 유효한 데이터를 샘플링하고,
        해당 컬럼에 대한 카테고리 추천 결과를 JSON 형식으로 반환합니다.
        """
        recommendations: Dict[str, Any] = {}

        # 유효 데이터가 존재하는 컬럼만 처리
        valid_columns = [col for col in self.df.columns if not self.df[col].dropna().empty]

        for column in tqdm_asyncio(valid_columns, desc="컬럼 처리 중", total=len(valid_columns)):
            try:
                sample_data = self._get_valid_samples(self.df[column])
                if not sample_data:
                    continue

                total_count = len(self.df[column])
                valid_count = len(self.df[column].dropna())
                valid_ratio = (valid_count / total_count) * 100
                logger.info(f"'{column}' 컬럼: 전체 {total_count}개 중 유효 데이터 {valid_count}개 ({valid_ratio:.1f}%)")

                prompt = self._generate_prompt_for_column(column, sample_data, valid_count, total_count)
                col_name, result = await self._get_recommendation(column, prompt)
                if result:
                    recommendations[col_name] = result
                    logger.info(f"'{column}' 컬럼 처리 완료")
            except Exception as e:
                logger.exception(f"'{column}' 컬럼 처리 중 오류")
                continue

            # 컬럼별 API 호출 후 추가 대기 (rate limit)
            await asyncio.sleep(Config.RATE_LIMIT_DELAY)

        return recommendations

    def _generate_prompt_for_column(self, column_name: str, sample_data: List[Any],
                                    valid_count: int, total_count: int) -> str:
        """
        각 컬럼에 대한 통계와 샘플 데이터를 포함하여 카테고리 추천 요청 프롬프트를 생성합니다.
        """
        data_stats = (
            f"전체 데이터 수: {total_count}개\n"
            f"유효 데이터 수: {valid_count}개 ({(valid_count/total_count*100):.1f}%)\n"
            f"현재 샘플 수: {len(sample_data)}개"
        )

        joined_data = "\n".join([str(item) for item in sample_data])
        prompt = f"""
        의료 데이터 분석가로서, '{column_name}' 컬럼의 데이터를 분석하여 
        의미 있는 카테고리를 추천해주세요.

        컬럼 통계:
        {data_stats}

        샘플 데이터:
        {joined_data}

        이 데이터의 특성을 반영하는 3-5개의 주요 카테고리를 JSON 형식으로 추천해주세요.
        각 카테고리는 "name"과 "description"을 포함해야 합니다.

        응답 형식:
        {{
          "categories": [
            {{"name": "카테고리명", "description": "상세 설명"}},
            {{"name": "카테고리명", "description": "상세 설명"}}
          ]
        }}
        """
        return prompt.strip()


async def main(api_key: str, df: pd.DataFrame) -> pd.DataFrame:
    """
    전체 DataFrame에 대해 각 컬럼의 카테고리 추천을 요청하고, 
    추천 결과를 CSV와 JSON 파일로 저장한 후 DataFrame으로 반환합니다.
    """
    recommender = MedicalCategoryRecommender(api_key, df)
    recommendations = await recommender.recommend_categories_for_all_columns()

    # 추천 결과를 DataFrame으로 변환
    categories_data: List[Dict[str, Any]] = []
    for column, result in recommendations.items():
        if 'categories' in result:
            for category in result['categories']:
                categories_data.append({
                    'column': column,
                    'category_name': category.get('name', ''),
                    'description': category.get('description', '')
                })

    categories_df = pd.DataFrame(categories_data)
    timestamp = time.strftime("%Y%m%d_%H%M%S")

    # CSV 저장
    csv_filename = f'category_recommendations_{timestamp}.csv'
    categories_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
    logger.info(f"추천 결과가 {csv_filename}에 저장되었습니다.")

    # JSON 저장
    json_filename = f'category_recommendations_{timestamp}.json'
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(recommendations, f, ensure_ascii=False, indent=2)
    logger.info(f"원본 추천 결과가 {json_filename}에 저장되었습니다.")

    return categories_df


if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()

    # 실제 실행 시, 'api_key'와 'df' (pandas DataFrame)를 알맞게 할당해야 합니다.
    # 예: api_key = os.environ.get("API_KEY")
    #     df = pd.read_csv("your_data.csv")


    try:
        results_df = asyncio.run(main(api_key, df))
        print("\n=== 처리 완료 ===")
        print(results_df)
    except Exception as e:
        logger.exception("메인 실행 중 치명적 오류 발생")


컬럼 처리 중:   0%|          | 0/26 [00:00<?, ?it/s]INFO:__main__:'CC' 컬럼: 전체 28108개 중 유효 데이터 28107개 (100.0%)
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:__main__:'CC' 컬럼 처리 완료
컬럼 처리 중:   4%|▍         | 1/26 [00:38<15:55, 38.21s/it]INFO:__main__:'약' 컬럼: 전체 28108개 중 유효 데이터 5893개 (21.0%)
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:__main__:'약' 컬럼 처리 완료
컬럼 처리 중:   8%|▊         | 2/26 [01:05<12:43, 31.83s/it]INFO:__main__:'장치 ' 컬럼: 전체 28108개 중 유효 데이터 11158개 (39.7%)
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:__main__:'장치 ' 컬럼 처리 완료
컬럼 처리 중:  12%|█▏        | 3/26 [01:34<11:43, 30.61s/it]INFO:__main__:'습관' 컬럼: 전체 28108개 중 유효 데이터 16002개 (56.9%)
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:__main__:'습관' 컬럼 처리 완료
컬럼 처리 중:  15%|█▌        | 4/26 [02:01<10:38, 29.03s/it]INFO:__main__:'찜질 ' 컬럼: 전체 28108개 중 유효 데이터 16886개


=== 처리 완료 ===
    column category_name                                        description
0       CC            증상  환자가 호소하는 주요 증상으로 통증, 불편감, 관절음, 개구제한 등이 포함됩니다. ...
1       CC            병력  턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기 등을 포함합니다. 교정...
2       CC    습관 및 생활 패턴  이갈이, 이악물기, 편측저작 등의 구강악습관과 수면, 스트레스 정도 등 생활 습관을...
3       CC    치료 계획 및 경과  물리치료, 약물요법, 장치치료, 보톡스 주사 등 계획된 치료 방법과 치료 경과 및 ...
4       CC      영상 검사 결과  파노라마, 측두하악관절 방사선 사진, CT 등 영상 검사 결과에 대한 판독 소견을 ...
..     ...           ...                                                ...
114   치료계획          물리치료                                물리치료를 통한 증상 완화 및 관리
115   치료계획         장치 관리                           교정장치, 스플린트 등의 장치 체크 및 조정
116   치료계획         습관 조절                           이갈이, 악습관 등의 습관 교정을 위한 지도
117   치료계획      약물/주사 치료                       MMTT, 보톡스, PDRN 등의 약물이나 주사요법
118   치료계획         경과 관찰                 증상 체크, 방사선 촬영 등을 통한 주기적 검진 및 경과 확인

[119 rows x 3 columns]


In [34]:
results_df

,column,category_name,description
0,CC,증상,"환자가 호소하는 주요 증상으로 통증, 불편감, 관절음, 개구제한 등이 포함됩니다. ..."
1,CC,병력,"턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기 등을 포함합니다. 교정..."
2,CC,습관 및 생활 패턴,"이갈이, 이악물기, 편측저작 등의 구강악습관과 수면, 스트레스 정도 등 생활 습관을..."
3,CC,치료 계획 및 경과,"물리치료, 약물요법, 장치치료, 보톡스 주사 등 계획된 치료 방법과 치료 경과 및 ..."
4,CC,영상 검사 결과,"파노라마, 측두하악관절 방사선 사진, CT 등 영상 검사 결과에 대한 판독 소견을 ..."
...,...,...,...
114,치료계획,물리치료,물리치료를 통한 증상 완화 및 관리
115,치료계획,장치 관리,"교정장치, 스플린트 등의 장치 체크 및 조정"
116,치료계획,습관 조절,"이갈이, 악습관 등의 습관 교정을 위한 지도"
117,치료계획,약물/주사 치료,"MMTT, 보톡스, PDRN 등의 약물이나 주사요법"


In [11]:
# pd.set_option('display.max_rows', None)
categories_df.to_excel('../data/category.xlsx')

NameError: name 'categories_df' is not defined